# 🎨 StockGen AI - Quality Comparison: x4plus vs x4v3
สมุดโน้ตเล่มนี้ใช้สำหรับเปรียบเทียบคุณภาพผลลัพธ์ (Visual Quality) ระหว่างโมเดลระดับพรีเมียม **x4plus** และโมเดลความเร็วสูง **x4v3** เพื่อตัดสินใจในการแบ่ง Tier ผู้ใช้ครับ

### 🚀 วิธีใช้งาน:
1. เลือก Runtime เป็น **GPU**
2. กด **Runtime** -> **Run all**
3. สังเกตความแตกต่างของความคมชัดและเวลาที่ใช้ด้านล่างสุดครับ

In [ ]:
# 📦 1. ติดตั้งไลบรารีและ Patch ระบบ
!pip install realesrgan gfpgan basicsr opencv-python matplotlib
import sys, os, types, torch, time, cv2, numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F_tv

try:
    shim = types.ModuleType("torchvision.transforms.functional_tensor")
    shim.rgb_to_grayscale = F_tv.rgb_to_grayscale
    sys.modules["torchvision.transforms.functional_tensor"] = shim
except: pass

from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
from basicsr.archs.srvgg_arch import SRVGGNetCompact

print("✅ Environment Ready!")

In [ ]:
# 🧠 2. ฟังก์ชันโหลดโมเดล
def load_x4plus():
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
    return RealESRGANer(scale=4, model_path=url, model=model, tile=0, half=True, device='cuda')

def load_x4v3():
    # v3 ใช้โครงสร้าง SRVGGNetCompact ที่เล็กกว่ามาก
    model = SRVGGNetCompact(num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=32, upscale=4, act_type='prelu')
    url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth'
    return RealESRGANer(scale=4, model_path=url, model=model, tile=0, half=True, device='cuda')

print("⏳ กำลังโหลดโมเดล (อาจใช้เวลาสักครู่ในการดาวน์โหลดครั้งแรก)...")
upsampler_plus = load_x4plus()
upsampler_v3 = load_x4v3()
print("✅ Models Loaded!")

In [ ]:
# 🖼️ 3. ดาวน์โหลดภาพตัวอย่างทดสอบ
!wget -O test.jpg https://raw.githubusercontent.com/xinntao/Real-ESRGAN/master/inputs/00003.png
img = cv2.imread('test.jpg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# ครอปภาพบางส่วนเพื่อให้เห็นรายละเอียดชัดๆ (Zoom-in)
h, w, _ = img.shape
img_crop = img[h//4:h//2, w//4:w//2]

print(f"📸 ขนาดภาพทดสอบ (Crop): {img_crop.shape[1]}x{img_crop.shape[0]}")

In [ ]:
# 🚀 4. รันการเปรียบเทียบ
print("⚡ กำลังประมวลผลด้วย x4plus...")
start = time.time()
out_plus, _ = upsampler_plus.enhance(img_crop, outscale=4)
time_plus = time.time() - start

print("⚡ กำลังประมวลผลด้วย x4v3...")
start = time.time()
out_v3, _ = upsampler_v3.enhance(img_crop, outscale=4)
time_v3 = time.time() - start

# 📊 5. แสดงผลเปรียบเทียบ
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

axes[0].imshow(out_plus)
axes[0].set_title(f"Premium: x4plus\nTime: {time_plus:.2f}s", fontsize=15)
axes[0].axis('off')

axes[1].imshow(out_v3)
axes[1].set_title(f"Standard: x4v3 (Tiny)\nTime: {time_v3:.2f}s", fontsize=15)
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"\n📈 สรุป: v3 เร็วกว่า x4plus ประมาณ {time_plus/time_v3:.1f} เท่า!")